# Vendor analysis: models V1, V2, V3

Delhivery cost-to-serve capstone. Data: vendor freight file, 296,946 consignment (LR) rows, of which 560 exact duplicates are dropped (see below), April 2025 to March 2026, vendor names anonymised.

| Task | Business question | Grain | Type |
|---|---|---|---|
| **V1** | What *should* this truck movement cost, given only the lane and the load? | shipment | regression |
| **V2** | Which consignments will lose money, before the vendor is paid? | LR | classification |
| **V3** | Which lane and vehicle choices end up below 60% full? | shipment | classification |

Every task uses the same rules: **time-based split** (train Apr-Dec 2025, test Jan-Mar 2026), **no leakage columns**, three algorithms plus a naive baseline, MAE/MAPE/R2 or PR-AUC as the headline metric, SHAP on the winning tree model. Hyper-parameters are fixed sensible defaults (no search); the Oct-Dec 2025 validation slice is used to choose classification thresholds.

In [1]:
import sys, os, warnings
sys.path.insert(0, os.path.abspath(".."))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, joblib
from IPython.display import display, Markdown
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)
from sklearn.metrics import average_precision_score, precision_recall_curve
from src import prep, evaluate as ev, plotting as pl
from src import vendor_pipeline as vp
from backend import vendor_codes
from src.modeling import build_regressor
CR = 1e7
fmt_cr = lambda x: f"₹{x/CR:,.1f} Cr"

## Load, clean, build the shipment table

The raw vendor file was converted once to `data/raw/vendor_lr.parquet`. Column names are normalised (`R2 Frieght` becomes `r2_freight`). Money fields: `gta_invoice_value` = billed to customer, `r2_freight` = paid to vendor, `margin` = billed - paid (asserted below). Rows are dropped if `vendor_name`, `route` or `lr_date` is missing; a missing `origin_plant` becomes `"UNKNOWN"`. One `shipment_no` is one truck; the shipment table (`data/processed/vendor_shipments.parquet`) aggregates LRs to the truck.

In [2]:
raw = prep.load()
print("raw LR rows in the file:", len(raw), "| columns:", raw.shape[1])
# exact duplicates (all columns equal) are dropped before any table is built; the dropped copies are saved for audit
raw, dups, dup_summary = prep.dedupe(raw)
print(f"exact duplicate rows (all {dup_summary['columns_compared']} columns): {dup_summary['duplicate_rows_dropped']:,} dropped, "
      f"carrying {fmt_cr(dup_summary['r2_freight_rs_in_duplicates'])} of freight across {dup_summary['shipments_affected']} shipments")
print("saved: data/processed/audit_duplicate_rows.csv (+ _summary.json) | LR rows after de-duplication:", len(raw))
dup_note = (f"{dup_summary['duplicate_rows_dropped']} exact duplicate rows (all {dup_summary['columns_compared']} columns equal; "
            f"{fmt_cr(dup_summary['r2_freight_rs_in_duplicates'])} of freight, {dup_summary['shipments_affected']} shipments) were dropped before modelling; see data/processed/audit_duplicate_rows.csv.")
assert np.allclose(raw["margin"], raw["gta_invoice_value"] - raw["r2_freight"], atol=1), "margin identity broken"
print("margin == gta_invoice_value - r2_freight: OK")
print("dropped for missing vendor/route/date:", len(raw) - len(prep.clean(raw)))
lr, ship = prep.build(raw)
print("clean LR rows:", len(lr), "| shipments:", len(ship))

raw LR rows in the file: 296946 | columns: 69


exact duplicate rows (all 69 columns): 560 dropped, carrying ₹1.1 Cr of freight across 322 shipments
saved: data/processed/audit_duplicate_rows.csv (+ _summary.json) | LR rows after de-duplication: 296386
margin == gta_invoice_value - r2_freight: OK


dropped for missing vendor/route/date: 1147


clean LR rows: 295239 | shipments: 172826


### Evidence for the headline figures (filters spelled out)

Two numbers drive the "Corrections" table: truck utilisation and margin %. Each depends on an exact filter and denominator, so both variants are shown side by side.

In [3]:
h = prep.headline_figures(lr, ship)
vc = ship[ship["vehicle_capacity"] > 0]
print("UTILISATION  (shipment table, one row per truck)")
print(f"  all trucks with capacity > 0                 n={len(vc):>7,}  median = {vc['utilisation'].median():.2f}%")
print(f"  ... and utilisation <= 150% (reported filter)   n={len(vc)-h['n_util_outliers']:>7,}  median = {h['util_median']:.2f}%   <- reported")
print(f"  trucks above 150% (data errors, excluded)    n={h['n_util_outliers']:>7,}")
print(f"  share of trucks below 60%: {h['util_below60']:.2f}%")
print()
print("MARGIN  (LR level; valid_cost = r2_freight > 0 and gta_invoice_value > 0, after dropping missing vendor/route/date)")
print(f"  valid-cost LRs n={h['n_lr_valid_cost']:,}  margin = {fmt_cr(h['margin_cr']*CR)}  billed = {fmt_cr(h['gta_cr']*CR)}  freight paid = {fmt_cr(h['freight_cr']*CR)}")
print(f"  margin / billed value (gta_invoice_value) = {h['margin_pct_gta']:.2f}%   <- reported")
print(f"  margin / freight paid (r2_freight)        = {h['margin_pct_freight']:.2f}%   <- not used")
print(f"  LRs with negative margin: {h['neg_share']:.1f}%  lost {fmt_cr(h['neg_cr']*CR)}")
display(Markdown(prep.corrections(lr, ship)))

UTILISATION  (shipment table, one row per truck)


  all trucks with capacity > 0                 n=161,985  median = 92.74%
  ... and utilisation <= 150% (reported filter)   n=161,438  median = 92.65%   <- reported
  trucks above 150% (data errors, excluded)    n=    547
  share of trucks below 60%: 9.12%

MARGIN  (LR level; valid_cost = r2_freight > 0 and gta_invoice_value > 0, after dropping missing vendor/route/date)
  valid-cost LRs n=284,421  margin = ₹24.4 Cr  billed = ₹563.0 Cr  freight paid = ₹538.6 Cr
  margin / billed value (gta_invoice_value) = 4.34%   <- reported
  margin / freight paid (r2_freight)        = 4.53%   <- not used
  LRs with negative margin: 31.9%  lost ₹17.8 Cr


## Corrections to Phase 3 figures

| # | Phase 3 claim | Recomputed |
|---|---|---|
| 1 | 57.7-60% of trucks 'poorly utilised' (each LR measured alone) | truck-level utilisation (sum of `lr_qty` / `vehicle_capacity`, capacity > 0, utilisation <= 150%) median **92.6%** on 161,438 trucks; only **9.1%** below 60%. 547 trucks above 150% treated as data errors |
| 2 | Fixed Rs78/km cut-off per LR | truck-level freight per km median **Rs92/km** (valid-cost trucks) |
| 3 | Modelled imputed freight | LRs with `r2_freight` = 0 (11,985) or `gta_invoice_value` = 0 (1,187) are excluded from all models |
| 4 | Margin about 4.3% | on 284,421 valid-cost LRs, margin = **4.34%** of billed value (Rs24.4 Cr on Rs563.0 Cr billed; the Phase 3 figure divides by billed value, dividing by freight paid instead would give 4.53%); **31.9%** of LRs negative (Rs17.8 Cr lost) |
| 5 | - | rows with missing `vendor_name`, `route` or `lr_date` dropped; missing `origin_plant` labelled UNKNOWN |


## Split and leakage rules (all tasks)

* **Train:** `lr_date` 2025-04-01 to 2025-12-31. **Test:** 2026-01-01 to 2026-03-31. Never a random split.
* **Validation slice** (threshold selection): Oct-Dec 2025, scored by a copy of each model trained on Apr-Sep only. The reported model is then trained on the full training period and uses that threshold.
* **Never features:** `r2_freight, gta_invoice_value, margin, margin_pct, transport_charges_paid, total_transport_charges, other_charges, liability_amount, liability_amount_as_on_date, b2b_billing, shipment_cost_no, shipment_cost, condition_flow, reporting_number, rp5_shipment, billing_document_date, billed_qty, billing_qty_in_mt, ftl_value, propotionate_ftl_value` and the ID columns `delivery_id, shipment_no, container_id`.
* High-cardinality categoricals: levels with fewer than 20 training rows become `"OTHER"`; Ridge and Logistic use `TargetEncoder(cv=5)` fitted inside the pipeline (training folds only), RandomForest uses integer codes, LightGBM uses native categoricals.
* RandomForest is fitted on a random 50,000-row sample of the training rows when training is larger (runtime rule).

In [4]:
from src.features import LEAK_COLS, ID_COLS
VAL_START = pd.Timestamp("2025-10-01")
train_s, test_s = ship["train"], ship["test"]
print("shipments train / test:", train_s.sum(), "/", test_s.sum())
print("LR rows train / test:", (lr.lr_date <= prep.TRAIN_END).sum(), "/", (lr.lr_date >= prep.TEST_START).sum())

shipments train / test:

 132729 / 40097
LR rows train / test: 225368 / 69871


# V1: Fair-price ("should-cost") model

**Business question.** What should this truck movement cost, given only the lane and the load? A vendor whose actual price is consistently above this fair price is a renegotiation candidate.

* Rows: shipments where every LR has a real cost, `distance > 0`, `total_qty > 0`.
* Target: `freight` in ₹. Models are trained on `log1p(freight)` and scored in ₹ after converting back.
* **Vendor is excluded on purpose**: the model learns the market price for the lane, so a vendor's premium appears in the residual.
* Naive baseline: training median of freight per km multiplied by distance.

In [5]:
V1_CATS = ["vehicle_type", "truck_load", "zone", "state_destination", "origin_plant", "route", "destination_point", "shipment_type", "inco_terms"]
V1_NUMS = ["distance", "total_qty", "vehicle_capacity", "n_lrs", "n_routes", "month", "weekday"]
v1 = ship[ship["all_cost_valid"] & (ship["distance"] > 0) & (ship["total_qty"] > 0) & (ship["freight"] > 0)].copy().reset_index(drop=True)
tr1, te1 = v1["train"], v1["test"]
print("V1 rows:", len(v1), "| train:", tr1.sum(), "| test:", te1.sum())
res1 = vp.run_regression(v1, V1_CATS, V1_NUMS, "freight", tr1, te1)
kpk = (v1.loc[tr1, "freight"] / v1.loc[tr1, "distance"]).median()
naive1 = ev.reg_metrics(v1.loc[te1, "freight"], kpk * v1.loc[te1, "distance"])
tab1 = pd.DataFrame({**res1["table"], "Naive (median ₹/km × distance)": naive1}).T
tab1["fit_seconds"] = pd.Series(res1["fit_s"])
display(tab1)
print(f"training median freight per km = ₹{kpk:.1f}")

V1 rows: 166453 | train: 128938 | test: 37515


,MAE,RMSE,MAPE,R2,fit_seconds
Ridge,9825.00,21686.00,24.90,0.8051,0.6
RandomForest,2695.93,6691.31,11.29,0.9814,5.2
LightGBM,2730.40,6621.79,11.63,0.9818,5.0
Naive (median ₹/km × distance),10630.66,18376.48,51.10,0.8601,NaN


training median freight per km = ₹92.6


In [6]:
winner1, best1, ok1 = vp.pick_winner({m: res1["table"][m]["MAE"] for m in res1["table"]}, higher_is_better=False)
reason1 = (f"{winner1} chosen on MAE (₹{res1['table'][winner1]['MAE']:,.0f}). Models within 2% of the best MAE: {ok1}; "
           f"when several are within 2% the simplest is preferred.")
print("Winner:", winner1, "|", reason1)
pred1 = res1["pred"][winner1]
v1_te = v1[te1].copy()
v1_te["predicted"] = pred1
# 80% interval: LightGBM quantiles 0.1/0.9 fitted on Apr-Sep, split-conformal (CQR) calibrated on the Oct-Dec validation slice
early1, val1 = tr1 & (v1["lr_date"] < VAL_START), tr1 & (v1["lr_date"] >= VAL_START)
# one global correction Q under-covers small loads and over-covers big ones, so Q is calibrated per load band (Mondrian conformal; the global-Q interval is kept for comparison)
QTY_BINS, QTY_LABELS = [0, 5, 15, 25, np.inf], ["<5", "5-15", "15-25", ">25"]
band1 = pd.cut(v1["total_qty"], QTY_BINS, labels=QTY_LABELS, right=False).astype(str)
ci1 = vp.conformal_interval(v1, V1_CATS, V1_NUMS, "freight", early1, val1, te1, point_pred=pred1, groups=band1)
lo_m, hi_m = ci1["q_low"], ci1["q_high"]
v1_te["pi_low"], v1_te["pi_high"] = ci1["pi_low"], ci1["pi_high"]
cov = ci1["test_coverage_calibrated"]
yb, bt = v1_te["freight"].to_numpy(), band1[te1].to_numpy()
rows_b = []
for lab in QTY_LABELS:
    m_ = bt == lab
    inside = lambda a_, b_: float(((yb[m_] >= a_[m_]) & (yb[m_] <= b_[m_])).mean())
    rows_b.append({"band_MT": lab, "n_test": int(m_.sum()), "n_calibration": int((band1[val1] == lab).sum()), "Q_log_scale": ci1["Q_by_group"][lab],
                   "coverage_global_Q": inside(ci1["pi_low_global"], ci1["pi_high_global"]), "coverage_per_band_Q": inside(ci1["pi_low"], ci1["pi_high"]),
                   "median_width_rs_global_Q": float(np.median((ci1["pi_high_global"] - ci1["pi_low_global"])[m_])),
                   "median_width_rs_per_band_Q": float(np.median((ci1["pi_high"] - ci1["pi_low"])[m_])),
                   "median_width_pct_of_forecast_per_band_Q": float(np.median(((ci1["pi_high"] - ci1["pi_low"]) / pred1)[m_]) * 100)})
band_tab = pd.DataFrame(rows_b)
display(band_tab.style.format({"Q_log_scale": "{:.3f}", "coverage_global_Q": "{:.1%}", "coverage_per_band_Q": "{:.1%}", "median_width_rs_global_Q": "₹{:,.0f}",
                               "median_width_rs_per_band_Q": "₹{:,.0f}", "median_width_pct_of_forecast_per_band_Q": "{:.0f}%"}))
display(Markdown("**Reading the table.** With one global correction the 80% interval covers too few small loads and too many large ones. Calibrating Q separately for each load band (validation rows of that band only) moves every band toward 80%. Small loads get a wider interval relative to the forecast, large loads a narrower one."))
print(f"calibration rows (validation, Oct-Dec): {ci1['n_cal']:,} | raw quantile interval covers {ci1['val_coverage_raw']:.1%} of them; conformal correction Q = {ci1['Q']:.3f} on the log scale")
print(f"TEST coverage of the nominal 80% interval: raw {ci1['test_coverage_raw']:.1%} -> calibrated {cov:.1%} | mean width ₹{ci1['test_mean_width_raw']:,.0f} -> ₹{ci1['test_mean_width_calibrated']:,.0f}")
out = pd.DataFrame({"record_id": v1_te["shipment_no"], "actual": v1_te["freight"], "predicted": v1_te["predicted"],
                    "abs_error": (v1_te["freight"] - v1_te["predicted"]).abs(), "pi_low": v1_te["pi_low"], "pi_high": v1_te["pi_high"]})
out.to_csv("../data/processed/pred_V1.csv", index=False)
print("saved pred_V1.csv", out.shape)

Winner: RandomForest | RandomForest chosen on MAE (₹2,696). Models within 2% of the best MAE: ['RandomForest', 'LightGBM']; when several are within 2% the simplest is preferred.


,band_MT,n_test,n_calibration,Q_log_scale,coverage_global_Q,coverage_per_band_Q,median_width_rs_global_Q,median_width_rs_per_band_Q,median_width_pct_of_forecast_per_band_Q
0,<5,1001,1311,0.247,69.3%,78.3%,"₹1,697","₹3,400",98%
1,5-15,17414,20372,0.043,74.0%,79.0%,"₹1,385","₹1,543",24%
2,15-25,9268,10581,0.009,79.1%,74.4%,"₹5,632","₹5,089",27%
3,>25,9832,8682,0.003,88.6%,81.1%,"₹46,907","₹44,569",53%


**Reading the table.** With one global correction the 80% interval covers too few small loads and too many large ones. Calibrating Q separately for each load band (validation rows of that band only) moves every band toward 80%. Small loads get a wider interval relative to the forecast, large loads a narrower one.

calibration rows (validation, Oct-Dec): 40,946 | raw quantile interval covers 64.8% of them; conformal correction Q = 0.021 on the log scale
TEST coverage of the nominal 80% interval: raw 67.3% -> calibrated 78.4% | mean width ₹18,630 -> ₹19,324


saved pred_V1.csv (37515, 6)


### V1 vendor over-pricing

`overpay_pct = (actual - predicted) / predicted` on the test period. The tables list every vendor x route x vehicle_type with at least 10 test shipments; `excess ₹` is the sum of positive gaps only.

In [7]:
v1_te["overpay_pct"] = (v1_te["freight"] - v1_te["predicted"]) / v1_te["predicted"] * 100
v1_te["gap"] = v1_te["freight"] - v1_te["predicted"]
v1_te["excess"] = v1_te["gap"].clip(lower=0)
g = v1_te.assign(vehicle_type=v1_te["vehicle_type"].fillna("MISSING")).groupby(["vendor", "route", "vehicle_type"]).agg(
    shipments=("shipment_no", "size"), actual_rs=("freight", "sum"), fair_rs=("predicted", "sum"),
    excess_rs=("excess", "sum"), median_overpay_pct=("overpay_pct", "median")).reset_index()
g = g[g.shipments >= 10].sort_values("excess_rs", ascending=False)
g.to_csv("../data/processed/vendor_overpay.csv", index=False)
vs = v1_te.groupby("vendor").agg(shipments=("shipment_no", "size"), actual_rs=("freight", "sum"), fair_rs=("predicted", "sum"),
    excess_rs=("excess", "sum"), median_overpay_pct=("overpay_pct", "median")).reset_index().sort_values("excess_rs", ascending=False)
vs.to_csv("../data/processed/vendor_overpay_summary.csv", index=False)
excess_total = float(v1_te["excess"].sum())
print(f"Total excess over fair price, test quarter: {fmt_cr(excess_total)}  (indicative annualised x4: {fmt_cr(excess_total*4)})")
print("vendor x route x vehicle rows:", len(g), "| vendors:", len(vs))
display(vs.assign(vendor=vs["vendor"].map(vendor_codes.code_for)).head(8).round(1))

Total excess over fair price, test quarter: ₹4.0 Cr  (indicative annualised x4: ₹15.9 Cr)
vendor x route x vehicle rows: 785 | vendors: 109


,vendor,shipments,actual_rs,fair_rs,excess_rs,median_overpay_pct
19,V002,1606,120705883.2,116964494.6,7404233.5,1.5
20,V001,2959,192627025.9,192979911.5,2743252.2,0.1
99,V021,229,24854152.6,22893314.9,2127488.1,9.0
91,V027,152,24990649.7,22957704.1,2104949.4,9.2
106,V007,443,72717199.3,72894797.3,2031804.2,1.0
97,V003,3389,70702896.7,71075555.2,1755510.0,-0.0
76,V037,660,8208581.1,6867353.0,1694090.0,3.4
56,V005,559,53380883.9,52600945.7,1600850.6,1.1


In [8]:
tree1 = winner1 if winner1 != "Ridge" else min(["RandomForest", "LightGBM"], key=lambda m: res1["table"][m]["MAE"])
print("SHAP model:", tree1)
drivers1, (Xs1, Z1, sv1, b1), imp1 = vp.shap_report(
    res1["fitted"][tree1], v1[tr1][V1_CATS + V1_NUMS].sample(20000, random_state=42), V1_CATS, "V1",
    "Distance and load size explain most of the fair freight price", "Fair price rises with distance, load and vehicle size (SHAP, log ₹)",
    "fair freight (log ₹)", sample=2000 if tree1 == "LightGBM" else 300)
display(pd.DataFrame(drivers1))

SHAP model: RandomForest


,feature,mean_abs_shap,plain_english
0,distance,0.64120,Higher distance raises the predicted fair frei...
1,total_qty,0.35489,Higher total_qty raises the predicted fair fre...
2,vehicle_capacity,0.12978,Higher vehicle_capacity raises the predicted f...
3,zone,0.07299,zone: the biggest push toward fair freight (lo...
4,state_destination,0.05515,state_destination: the biggest push toward fai...


In [9]:
if tree1 != winner1:
    print(f"Note: SHAP drivers are from {tree1} because the winner ({winner1}) is not a tree model.")
joblib.dump({"pipeline": res1["fitted"][winner1], "q_low": lo_m, "q_high": hi_m, "conformal_Q": ci1["Q"], "conformal_Q_by_band": ci1["Q_by_group"], "qty_band_edges": QTY_BINS, "qty_band_labels": QTY_LABELS,
             "cats": V1_CATS, "nums": V1_NUMS, "target": "freight (₹, log1p inside)"},
            f"../models/V1_{winner1}.joblib", compress=3)
v1_best = res1["fitted"][winner1]
vp.save_json("V1", {
    "task_id": "V1", "problem": "vendor", "type": "regression",
    "business_question": "What should this truck movement cost, given only the lane and the load?",
    "target": "freight (₹ per shipment; models trained on log1p)", "grain": "shipment",
    "train_period": "2025-04-01 to 2025-12-31", "test_period": "2026-01-01 to 2026-03-31",
    "n_train": int(tr1.sum()), "n_test": int(te1.sum()),
    "base_rate_or_baseline": {"metric": "training median ₹/km × distance (MAE)", "value": naive1["MAE"], "all_metrics": naive1, "median_rs_per_km": float(kpk)},
    "features": V1_CATS + V1_NUMS,
    "excluded_for_leakage": LEAK_COLS + ID_COLS + ["vendor (deliberately excluded: vendor effect is read from the residual)"],
    "models": [{"name": m, "metrics": res1["table"][m], "fit_seconds": res1["fit_s"][m]} for m in res1["table"]],
    "winner": winner1, "winner_reason": reason1, "threshold": None, "top_drivers": drivers1,
    "business_impact_inputs": {"label": "indicative", "test_total_freight_rs": float(v1_te.freight.sum()),
        "test_excess_over_fair_rs": excess_total, "annualised_excess_rs_x4": excess_total * 4,
        "interval_80pct_coverage": float(cov), "interval_coverage_before_calibration": ci1["test_coverage_raw"],
        "conformal": {k: ci1[k] for k in ("Q", "n_cal", "target_coverage", "val_coverage_raw", "test_coverage_raw", "test_coverage_calibrated", "test_mean_width_raw", "test_mean_width_calibrated")},
        "interval_by_total_qty_band": band_tab.to_dict("records"), "vendors": int(len(vs))},
    "caveats": [dup_note, "The 'fair price' is the market median for this lane profile in this vendor pool; it is not the vendor's true cost.",
                "RandomForest is fitted on a 50,000-row training sample; Ridge and LightGBM use all training rows.",
                "Hyper-parameters are fixed defaults (no search).", f"pi_low/pi_high: LightGBM quantiles (0.1/0.9) calibrated by split-conformal (CQR) on the Oct-Dec 2025 validation slice, with a separate correction per total_qty band (<5, 5-15, 15-25, >25 MT; global Q = {ci1['Q']:.3f} on the log scale); test coverage of the nominal 80% interval is {cov:.1%} (raw quantiles: {ci1['test_coverage_raw']:.1%}). Coverage is only guaranteed if test data behave like the validation slice.", "One client sector (polyester), one year of data, anonymised vendors."]})
display(Markdown(f"**V1 CHECKPOINT data saved.** Winner: **{winner1}**."))

**V1 CHECKPOINT data saved.** Winner: **RandomForest**.

# V2: Loss-making consignment risk (classification, one row per LR)

**Business question.** At booking, before the vendor is paid, which consignments will lose money? The action is to reassign the vendor or reprice before dispatch.

* Rows: `valid_cost` LRs. Target: `margin < 0` (about 32% of LRs).
* Features: vendor, lane, vehicle, quantity, billing and calendar fields, plus two columns from the shipment table (`n_lrs`, `shipment_utilisation`). No money fields are used.
* Extra baseline, the **rate-card lookup**: historical loss rate of `route x vendor x vehicle_type` in training (global training rate when the combination has fewer than 5 rows).
* Threshold rule: the threshold that maximises recall while keeping precision >= 0.60 **on the validation slice only** (Oct-Dec 2025, scored by a copy of the model trained on Apr-Sep), applied unchanged to the test quarter. Test labels play no part in choosing it.

In [10]:
V2_CATS = ["vendor_name", "route", "origin_plant", "destination_point", "zone", "state_destination", "vehicle_type", "truck_load",
           "billed_uom", "inco_terms", "shipment_type", "billing_type"]
V2_NUMS = ["vehicle_capacity", "lr_qty", "distance", "month", "weekday", "n_lrs", "shipment_utilisation"]
v2 = lr[(lr["r2_freight"] > 0) & (lr["gta_invoice_value"] > 0)].copy()
v2["month"], v2["weekday"] = v2["lr_date"].dt.month, v2["lr_date"].dt.weekday
v2 = v2.merge(ship[["shipment_no", "n_lrs", "utilisation"]].rename(columns={"utilisation": "shipment_utilisation"}), on="shipment_no", how="left")
v2["loss"] = (v2["margin"] < 0).astype(int)
v2 = v2.reset_index(drop=True)
tr2, te2 = (v2.lr_date <= prep.TRAIN_END), (v2.lr_date >= prep.TEST_START)
print("V2 rows:", len(v2), "| train:", tr2.sum(), "| test:", te2.sum(), f"| loss base rate train {v2.loc[tr2,'loss'].mean():.3f}, test {v2.loc[te2,'loss'].mean():.3f}")
res2 = vp.run_classification(v2, V2_CATS, V2_NUMS, "loss", tr2, te2, VAL_START)
y2 = v2.loc[te2, "loss"].values
# Threshold: chosen on the Oct-Dec 2025 validation slice only (max recall with precision >= 0.60, models trained on Apr-Sep), then applied unchanged to test
print({m: {"thr": round(res2["thr"][m], 4), "val_P": res2["table"][m]["val_precision_at_thr"], "val_R": res2["table"][m]["val_recall_at_thr"],
           "test_P": res2["table"][m]["precision"], "test_R": res2["table"][m]["recall"]} for m in res2["thr"]})

V2 rows: 284421 | train: 219109 | test: 65312 | loss base rate train 0.324, test 0.304


{'Logistic': {'thr': 0.717, 'val_P': 0.6, 'val_R': 0.3535, 'test_P': 0.6264, 'test_R': 0.5712}, 'RandomForest': {'thr': 0.4904, 'val_P': 0.6, 'val_R': 0.7103, 'test_P': 0.6074, 'test_R': 0.7989}, 'LightGBM': {'thr': 0.5345, 'val_P': 0.6, 'val_R': 0.7407, 'test_P': 0.5967, 'test_R': 0.8391}}


In [11]:
y2 = v2.loc[te2, "loss"].values
rate2, base2, _ = vp.lookup_rate(v2[tr2], v2[te2], ["route", "vendor_name", "vehicle_type"], "loss")
lookup2 = round(float(average_precision_score(y2, rate2)), 4)
prev2 = round(float(y2.mean()), 4)
tab2 = pd.DataFrame(res2["table"]).T
tab2.loc["Rate-card lookup (route×vendor×vehicle)"] = {"PR_AUC": lookup2}
tab2.loc["Base rate (train rate, constant)"] = {"PR_AUC": prev2}
tab2["fit_seconds"] = pd.Series(res2["fit_s"])
display(tab2)

,PR_AUC,ROC_AUC,precision,recall,F1,threshold,val_precision_at_thr,val_recall_at_thr,thr_rule,fit_seconds
Logistic,0.6617,0.8093,0.6264,0.5712,0.5975,0.717,0.6,0.3535,recall-max with val precision>=0.60,3.1
RandomForest,0.6991,0.8632,0.6074,0.7989,0.6901,0.4904,0.6,0.7103,recall-max with val precision>=0.60,5.1
LightGBM,0.7429,0.874,0.5967,0.8391,0.6975,0.5345,0.6,0.7407,recall-max with val precision>=0.60,7.2
Rate-card lookup (route×vendor×vehicle),0.5814,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
"Base rate (train rate, constant)",0.3042,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [12]:
winner2, best2, ok2 = vp.pick_winner({m: res2["table"][m]["PR_AUC"] for m in res2["table"]}, higher_is_better=True)
gap2 = res2["table"]["LightGBM"]["PR_AUC"] - lookup2
reason2 = (f"{winner2} chosen on PR-AUC ({res2['table'][winner2]['PR_AUC']:.4f}); models within 2% of the best: {ok2} (simplest preferred).")
print("Winner:", winner2, "|", reason2)
print(f"LightGBM PR-AUC {res2['table']['LightGBM']['PR_AUC']:.4f} - rate-card lookup PR-AUC {lookup2:.4f} = {gap2:+.4f}")
if gap2 < 0.03:
    conclusion2 = "Losses are mostly fixed by the lane–vendor rate combination, so the fix is renegotiating the rate card, not prediction."
    display(Markdown(f"> **V2 conclusion.** LightGBM beats the rate-card lookup by only {gap2:.3f} PR-AUC (< 0.03). *{conclusion2}*"))
else:
    conclusion2 = (f"LightGBM beats the rate-card lookup by {gap2:.3f} PR-AUC (>= 0.03), so the model adds signal beyond the lane-vendor rate card "
                   "so the 'renegotiate the rate card, not prediction' conclusion does not apply.")
    display(Markdown(f"> **V2 conclusion.** {conclusion2}"))

Winner: LightGBM | LightGBM chosen on PR-AUC (0.7429); models within 2% of the best: ['LightGBM'] (simplest preferred).
LightGBM PR-AUC 0.7429 - rate-card lookup PR-AUC 0.5814 = +0.1615


> **V2 conclusion.** LightGBM beats the rate-card lookup by 0.161 PR-AUC (>= 0.03), so the model adds signal beyond the lane-vendor rate card so the 'renegotiate the rate card, not prediction' conclusion does not apply.

**Sensitivity check.** `n_lrs` and `shipment_utilisation` describe the whole truck, which may not be known at booking. The cell below refits LightGBM without them to see whether the lead over the rate-card lookup depends on them.

In [13]:
from src.modeling import build_classifier
N2b = [c for c in V2_NUMS if c not in ("n_lrs", "shipment_utilisation")]
m2b = build_classifier("LightGBM", V2_CATS, N2b).fit(v2.loc[tr2, V2_CATS + N2b], v2.loc[tr2, "loss"])
sens2 = round(float(average_precision_score(y2, m2b.predict_proba(v2.loc[te2, V2_CATS + N2b])[:, 1])), 4)
print(f"LightGBM without n_lrs / shipment_utilisation: PR-AUC {sens2:.4f} vs rate-card lookup {lookup2:.4f} (gap {sens2 - lookup2:+.4f})")

LightGBM without n_lrs / shipment_utilisation: PR-AUC 0.6730 vs rate-card lookup 0.5814 (gap +0.0916)


In [14]:
thr2 = res2["thr"][winner2]; p2 = res2["proba"][winner2]
lab2 = (p2 >= thr2).astype(int)
m2 = v2[te2].copy(); m2["flag"] = lab2
tp2 = m2[(m2.flag == 1) & (m2.loss == 1)]; missed2 = m2[(m2.flag == 0) & (m2.loss == 1)]
impact2 = {"flagged": int(lab2.sum()), "true_positives": int(len(tp2)), "actual_losses_in_test": int(m2.loss.sum()),
           "loss_on_true_positives_rs": float(tp2.margin.abs().sum()), "loss_missed_rs": float(missed2.margin.abs().sum())}
print(res2["table"][winner2]["thr_rule"], "| threshold =", round(thr2, 4))
print({k: (fmt_cr(v) if k.endswith("_rs") else v) for k, v in impact2.items()})
pd.DataFrame({"record_id": m2["delivery_id"], "actual": m2["loss"], "predicted_label": lab2, "probability": p2, "threshold": thr2}).to_csv("../data/processed/pred_V2.csv", index=False)
pc, rc, tc = ev.pr_curve(y2, p2)
pd.DataFrame({"precision": pc[:-1], "recall": rc[:-1], "threshold": tc}).to_csv("../reports/metrics/V2_pr_curve.csv", index=False)

recall-max with val precision>=0.60 | threshold = 0.5345
{'flagged': 27940, 'true_positives': 16672, 'actual_losses_in_test': 19868, 'loss_on_true_positives_rs': '₹3.7 Cr', 'loss_missed_rs': '₹0.6 Cr'}


In [15]:
tree2 = winner2 if winner2 != "Logistic" else "LightGBM"
Xte2 = v2[te2][V2_CATS + V2_NUMS]
drivers2, (Xs2, Z2, sv2, b2), imp2 = vp.shap_report(res2["fitted"][tree2], Xte2, V2_CATS, "V2",
    "Lane, vendor and vehicle choices drive the risk of a loss-making consignment", "What pushes a consignment toward a loss (SHAP, log-odds)", "loss risk")
display(pd.DataFrame(drivers2))
# three example consignments from the SHAP sample: high-risk, borderline, safe
ps = res2["fitted"][tree2].predict_proba(Xs2)[:, 1]
idx = {"high_risk": int(np.argmax(ps)), "borderline": int(np.argmin(np.abs(ps - thr2))), "safe": int(np.argmin(ps))}
ex_rows = []
for k, i in idx.items():
    vp.waterfall_png(Z2, sv2, b2, i, f"../reports/figures/V2_waterfall_{k}.png", f"{k.replace('_',' ').title()} consignment: why the model scores it {ps[i]:.2f}")
    r = v2.loc[Xs2.index[i]]
    ex_rows.append({"case": k, "model_probability": round(float(ps[i]), 3), "route": r.route, "vendor": vendor_codes.code_for(r.vendor_name), "vehicle_type": r.vehicle_type, "lr_qty": r.lr_qty, "actual_loss": int(r.loss)})
display(pd.DataFrame(ex_rows))

,feature,mean_abs_shap,plain_english
0,shipment_utilisation,1.12770,Higher shipment_utilisation lowers the predict...
1,route,0.92158,route: the biggest push toward loss risk comes...
2,vehicle_type,0.43411,vehicle_type: the biggest push toward loss ris...
3,vendor_name,0.40449,vendor_name: the biggest push toward loss risk...
4,month,0.38958,"month matters, but not in one direction: its e..."


,case,model_probability,route,vendor,vehicle_type,lr_qty,actual_loss
0,high_risk,0.999,HZBAV1,V013,SB02,7.744,1
1,borderline,0.535,PGJAU1,V002,OB27,25.000,0
2,safe,0.002,HZPLD1,V008,OB29,2.412,0


### V2 limitations
* One client sector (polyester) and one year of data; seasonality cannot be separated from trend.
* Rate cards may change, which would make last year's lane-vendor losses a poor guide.
* Vendors are anonymised, so findings cannot be tied to named contracts.
* `shipment_utilisation` and `n_lrs` come from the shipment the LR rides on; if the load plan is not known at booking, drop them before using the model operationally.

In [16]:
joblib.dump({"pipeline": res2["fitted"][winner2], "threshold": thr2, "cats": V2_CATS, "nums": V2_NUMS}, f"../models/V2_{winner2}.joblib", compress=3)
vp.save_json("V2", {
    "task_id": "V2", "problem": "vendor", "type": "classification",
    "business_question": "At booking, before the vendor is paid, which consignments will lose money?",
    "target": "margin < 0", "grain": "LR", "train_period": "2025-04-01 to 2025-12-31", "test_period": "2026-01-01 to 2026-03-31",
    "n_train": int(tr2.sum()), "n_test": int(te2.sum()),
    "base_rate_or_baseline": {"metric": "PR-AUC of constant base rate (= test prevalence)", "value": prev2, "train_base_rate": round(base2, 4)},
    "extra_baselines": [{"name": "Rate-card lookup (route×vendor×vehicle_type, fitted on train, <5 rows -> train base rate)", "PR_AUC": lookup2}],
    "features": V2_CATS + V2_NUMS, "excluded_for_leakage": LEAK_COLS + ID_COLS,
    "models": [{"name": m, "metrics": {k: v for k, v in res2["table"][m].items() if k not in ("thr_rule",)}, "fit_seconds": res2["fit_s"][m]} for m in res2["table"]],
    "winner": winner2, "winner_reason": reason2 + " " + conclusion2, "threshold": thr2, "threshold_rule": res2["table"][winner2]["thr_rule"],
    "top_drivers": drivers2, "business_impact_inputs": {"label": "test period", **impact2},
    "caveats": [dup_note, "One client sector (polyester), one year of data.", "Rate cards may change.", "Anonymised vendors.",
                f"Threshold {thr2:.4f} was selected on the Oct-Dec 2025 validation slice (precision {res2['table'][winner2]['val_precision_at_thr']:.3f}, recall {res2['table'][winner2]['val_recall_at_thr']:.3f} there) and applied unchanged to test, where precision is {res2['table'][winner2]['precision']:.3f} and recall {res2['table'][winner2]['recall']:.3f}.",
                f"shipment_utilisation and n_lrs are shipment-level and may not be known at booking; without them LightGBM PR-AUC is {sens2} (rate-card lookup {lookup2}).",
                "RandomForest fitted on a 50,000-row training sample."]})

# V3: Under-loaded truck risk (classification, one row per shipment)

**Business question.** When planning, before loading, which lane and vehicle choices end up below 60% full? The action is to use a smaller vehicle or combine loads.

* Rows: `valid_capacity` shipments that are not `util_outlier` (utilisation > 150%). Target: `utilisation < 60` (about 9%), so classes are imbalanced: `class_weight="balanced"` and PR-AUC as the headline.
* Features are planning-time only: lane, plant, vendor, vehicle, calendar, distance. **`total_qty`, `n_lrs` and `n_routes` are not used**; they are the outcome of loading and would leak the answer.

In [17]:
V3_CATS = ["route", "origin_plant", "destination_point", "zone", "state_destination", "vehicle_type", "truck_load", "vendor", "shipment_type"]
V3_NUMS = ["vehicle_capacity", "month", "weekday", "distance"]
v3 = ship[ship["valid_capacity"] & ~ship["util_outlier"]].copy().reset_index(drop=True)
v3["under"] = (v3["utilisation"] < 60).astype(int)
tr3, te3 = v3["train"], v3["test"]
print("V3 rows:", len(v3), "| train:", tr3.sum(), "| test:", te3.sum(), f"| under-loaded base rate train {v3.loc[tr3,'under'].mean():.3f}, test {v3.loc[te3,'under'].mean():.3f}")
res3 = vp.run_classification(v3, V3_CATS, V3_NUMS, "under", tr3, te3, VAL_START)
y3 = v3.loc[te3, "under"].values

V3 rows: 161438 | train: 124252 | test: 37186 | under-loaded base rate train 0.092, test 0.089


### Leakage-free lookup baseline

The lookup baseline scores each test shipment with the under-load rate of its `route x vehicle_type` combination **estimated on training shipments only** (combinations with fewer than 5 training shipments fall back to the training base rate). Fitting the rates on test labels would inflate PR-AUC, so they are never fitted on test rows.

In [18]:
rate3, base3, g3 = vp.lookup_rate(v3[tr3], v3[te3], ["route", "vehicle_type"], "under", min_rows=5)
lookup3 = round(float(average_precision_score(y3, rate3)), 4)
# contrast only: same lookup fitted on the TEST labels (this is the leaky version, never reported as a result)
rate3_leak, _, _ = vp.lookup_rate(v3[te3], v3[te3], ["route", "vehicle_type"], "under", min_rows=5)
leaky3 = round(float(average_precision_score(y3, rate3_leak)), 4)
n_fallback = int(((v3[te3].assign(vehicle_type=v3[te3].vehicle_type.fillna("MISSING")).merge(
    g3.reset_index().rename(columns={"size": "n_tr"})[["route", "vehicle_type", "n_tr"]], on=["route", "vehicle_type"], how="left")["n_tr"].fillna(0)) < 5).sum())
print(f"train combos: {len(g3):,} | train base rate {base3:.4f} | test shipments using the fallback rate: {n_fallback:,} of {te3.sum():,}")
print(f"TRAIN-only lookup PR-AUC (reported): {lookup3:.4f}")
print(f"test-label lookup PR-AUC (leaky, for contrast only): {leaky3:.4f}")
prev3 = round(float(y3.mean()), 4)
tab3 = pd.DataFrame(res3["table"]).T
tab3.loc["Lookup route×vehicle_type (train-only)"] = {"PR_AUC": lookup3}
tab3.loc["Base rate (constant)"] = {"PR_AUC": prev3}
tab3["fit_seconds"] = pd.Series(res3["fit_s"])
display(tab3)

train combos: 7,763 | train base rate 0.0918 | test shipments using the fallback rate: 5,257 of 37,186
TRAIN-only lookup PR-AUC (reported): 0.6333
test-label lookup PR-AUC (leaky, for contrast only): 0.6906


,PR_AUC,ROC_AUC,precision,recall,F1,threshold,val_precision_at_thr,val_recall_at_thr,thr_rule,fit_seconds
Logistic,0.6616,0.9068,0.6558,0.5484,0.5973,0.8414,0.6001,0.5424,recall-max with val precision>=0.60,1.2
RandomForest,0.6899,0.9204,0.5417,0.6821,0.6038,0.4907,0.6001,0.6769,recall-max with val precision>=0.60,3.3
LightGBM,0.7516,0.9358,0.6554,0.6965,0.6753,0.6822,0.6,0.7196,recall-max with val precision>=0.60,4.4
Lookup route×vehicle_type (train-only),0.6333,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Base rate (constant),0.0891,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [19]:
winner3, best3, ok3 = vp.pick_winner({m: res3["table"][m]["PR_AUC"] for m in res3["table"]}, higher_is_better=True)
reason3 = f"{winner3} chosen on PR-AUC ({res3['table'][winner3]['PR_AUC']:.4f}); models within 2% of the best: {ok3} (simplest preferred)."
gap3 = res3["table"]["LightGBM"]["PR_AUC"] - lookup3
print("Winner:", winner3, "|", reason3)
print(f"LightGBM minus train-only lookup: {gap3:+.4f} PR-AUC")
thr3 = res3["thr"][winner3]; p3 = res3["proba"][winner3]
lab3 = (p3 >= thr3).astype(int)
print(res3["table"][winner3]["thr_rule"], "| threshold =", round(thr3, 4), "| flagged:", int(lab3.sum()), "of", len(lab3))
m3 = v3[te3].copy(); m3["flag"] = lab3
pd.DataFrame({"record_id": m3["shipment_no"], "actual": m3["under"], "predicted_label": lab3, "probability": p3, "threshold": thr3}).to_csv("../data/processed/pred_V3.csv", index=False)
pc, rc, tc = ev.pr_curve(y3, p3)
pd.DataFrame({"precision": pc[:-1], "recall": rc[:-1], "threshold": tc}).to_csv("../reports/metrics/V3_pr_curve.csv", index=False)

Winner: LightGBM | LightGBM chosen on PR-AUC (0.7516); models within 2% of the best: ['LightGBM'] (simplest preferred).
LightGBM minus train-only lookup: +0.1183 PR-AUC
recall-max with val precision>=0.60 | threshold = 0.6822 | flagged: 3523 of 37186


In [20]:
tree3 = winner3 if winner3 != "Logistic" else "LightGBM"
drivers3, _, imp3 = vp.shap_report(res3["fitted"][tree3], v3[te3][V3_CATS + V3_NUMS], V3_CATS, "V3",
    "Vehicle size and lane decide whether a truck leaves under-loaded", "What pushes a truck toward under-loading (SHAP, log-odds)", "under-loading risk")
display(pd.DataFrame(drivers3))

,feature,mean_abs_shap,plain_english
0,origin_plant,1.07348,origin_plant: the biggest push toward under-lo...
1,route,0.99918,route: the biggest push toward under-loading r...
2,destination_point,0.71273,destination_point: the biggest push toward und...
3,vehicle_type,0.45703,vehicle_type: the biggest push toward under-lo...
4,vendor,0.41523,vendor: the biggest push toward under-loading ...


### V1 counterfactual saving (indicative)

For every **test** shipment flagged by V3: on the same route, take the **smallest `vehicle_capacity` seen in training** that is >= the shipment's `total_qty`; set `vehicle_capacity` and the matching `vehicle_type` (the most common type used with that capacity on that route in training); re-price with the V1 pipeline; saving = `max(actual - counterfactual, 0)`.

Only shipments with a real actual freight (V1-eligible rows) can be priced; routes never seen in training, or with no training capacity large enough, are not covered. These are **indicative** numbers: V1 is a market-median price, not a quote.

In [21]:
flagged = m3[m3.flag == 1]
elig = flagged[flagged["all_cost_valid"] & (flagged["distance"] > 0) & (flagged["total_qty"] > 0) & (flagged["freight"] > 0)]
cf = vp.counterfactual_saving(elig, v3[tr3], v1_best, V1_CATS, V1_NUMS)
# vehicle-change-only effect: V1 price of the original spec minus V1 price of the counterfactual spec
vehicle_effect = float((cf["pred_orig"] - cf["pred_cf"]).clip(lower=0).sum())
cf_stats = {
    "label": "indicative",
    "flagged_in_test": int(len(flagged)), "flagged_with_real_freight": int(len(elig)),
    "covered_with_counterfactual": int(len(cf)), "covered_with_strictly_smaller_vehicle": int(cf["smaller_vehicle"].sum()),
    "shipments_with_positive_saving": int((cf["saving"] > 0).sum()),
    "total_saving_rs": float(cf["saving"].sum()),
    "median_saving_pct_all_covered": float(cf["saving_pct"].median()),
    "median_saving_pct_where_positive": float(cf.loc[cf["saving"] > 0, "saving_pct"].median()),
    "actual_freight_on_covered_rs": float(cf["freight"].sum()),
    "vehicle_change_only_effect_rs": vehicle_effect,
}
print(f"flagged {cf_stats['flagged_in_test']:,} -> with real freight {cf_stats['flagged_with_real_freight']:,} -> covered by a counterfactual {cf_stats['covered_with_counterfactual']:,} "
      f"(of which strictly smaller vehicle: {cf_stats['covered_with_strictly_smaller_vehicle']:,})")
print(f"total indicative saving: {fmt_cr(cf_stats['total_saving_rs'])}  on actual freight of {fmt_cr(cf_stats['actual_freight_on_covered_rs'])}")
print(f"median saving %: {cf_stats['median_saving_pct_all_covered']:.1f}% over all covered shipments; {cf_stats['median_saving_pct_where_positive']:.1f}% over the {cf_stats['shipments_with_positive_saving']:,} with a positive saving")
print(f"part of that saving due to the vehicle change alone (V1 price original spec - counterfactual spec): {fmt_cr(vehicle_effect)}")
display(cf[["saving", "saving_pct"]].describe(percentiles=[.1, .5, .9]).round(1))  # distribution only: no shipment-level rows are shown

flagged 3,523 -> with real freight 3,517 -> covered by a counterfactual 3,358 (of which strictly smaller vehicle: 2,416)
total indicative saving: ₹1.1 Cr  on actual freight of ₹6.4 Cr
median saving %: 4.1% over all covered shipments; 26.1% over the 1,885 with a positive saving
part of that saving due to the vehicle change alone (V1 price original spec - counterfactual spec): ₹1.1 Cr


,saving,saving_pct
count,3358.0,3358.0
mean,3345.0,13.6
std,5323.1,16.3
min,0.0,0.0
10%,0.0,0.0
50%,380.1,4.1
90%,9846.7,39.0
max,57639.5,84.1


In [22]:
under_test = m3[m3.under == 1]
impact3 = {"label": "test period; counterfactual is indicative",
           "flagged": int(lab3.sum()), "true_positives": int(((m3.flag == 1) & (m3.under == 1)).sum()), "actual_underloaded": int(len(under_test)),
           "freight_on_underloaded_rs": float(under_test["freight"].where(under_test["all_cost_valid"]).sum()),
           "avg_utilisation_underloaded_pct": float(under_test["utilisation"].mean()),
           "counterfactual_saving": cf_stats}
print({k: v for k, v in impact3.items() if k != "counterfactual_saving"})
joblib.dump({"pipeline": res3["fitted"][winner3], "threshold": thr3, "cats": V3_CATS, "nums": V3_NUMS}, f"../models/V3_{winner3}.joblib", compress=3)
vp.save_json("V3", {
    "task_id": "V3", "problem": "vendor", "type": "classification",
    "business_question": "When planning, before loading, which lane and vehicle choices end up below 60% full?",
    "target": "utilisation < 60", "grain": "shipment", "train_period": "2025-04-01 to 2025-12-31", "test_period": "2026-01-01 to 2026-03-31",
    "n_train": int(tr3.sum()), "n_test": int(te3.sum()),
    "base_rate_or_baseline": {"metric": "PR-AUC of constant base rate (= test prevalence)", "value": prev3, "train_base_rate": round(base3, 4)},
    "extra_baselines": [{"name": "Lookup route×vehicle_type, rates fitted on TRAIN shipments only (<5 rows -> train base rate)", "PR_AUC": lookup3,
                         "test_shipments_on_fallback_rate": n_fallback, "note": "rates fitted on training shipments only"}],
    "features": V3_CATS + V3_NUMS,
    "excluded_for_leakage": LEAK_COLS + ID_COLS + ["total_qty", "n_lrs", "n_routes (outcome of loading)"],
    "models": [{"name": m, "metrics": {k: v for k, v in res3["table"][m].items() if k not in ("thr_rule",)}, "fit_seconds": res3["fit_s"][m]} for m in res3["table"]],
    "winner": winner3, "winner_reason": reason3, "threshold": thr3, "threshold_rule": res3["table"][winner3]["thr_rule"],
    "top_drivers": drivers3, "business_impact_inputs": impact3,
    "caveats": [dup_note, "Planning-time features only; total_qty, n_lrs and n_routes excluded because they are the outcome of loading.",
                "Classes are imbalanced (about 9% under-loaded); judge on PR-AUC.",
                "Counterfactual saving is indicative: V1 is a market-median price, capacities come from training history on the same route, and shipments on routes without a suitable training capacity are not covered.",
                "RandomForest fitted on a 50,000-row training sample."]})

{'label': 'test period; counterfactual is indicative', 'flagged': 3523, 'true_positives': 2309, 'actual_underloaded': 3315, 'freight_on_underloaded_rs': 59061408.79000001, 'avg_utilisation_underloaded_pct': 48.42963658875718}


# Supporting analysis: audit list (unsupervised)

An Isolation Forest on shipment-level economics. With `contamination=0.02`, **2% of shipments are flagged by construction**; the list is a place to start an audit, not a measured error rate.

In [23]:
from sklearn.ensemble import IsolationForest
a = ship[ship["all_cost_valid"] & (ship["distance"] > 0) & (ship["total_qty"] > 0)].copy()
a["freight_per_km"] = a["freight"] / a["distance"]; a["freight_per_mt"] = a["freight"] / a["total_qty"]; a["margin_pct"] = a["margin"] / a["gta"] * 100
cols = ["distance", "total_qty", "vehicle_capacity", "freight", "freight_per_km", "freight_per_mt", "margin_pct", "utilisation"]
iso = IsolationForest(n_estimators=200, contamination=0.02, random_state=42, n_jobs=-1).fit(a[cols].fillna(a[cols].median()))
a["anomaly"] = iso.predict(a[cols].fillna(a[cols].median())) == -1
a[a.anomaly][["shipment_no", "vendor", "route", "lr_date"] + cols].to_csv("../data/processed/audit_list_isolation_forest.csv", index=False)
print("flagged:", int(a.anomaly.sum()), "of", len(a), "(2% by construction)")
display(a[a.anomaly].assign(vendor=a["vendor"].map(vendor_codes.code_for)).groupby("vendor").size().sort_values(ascending=False).head(5).rename("flagged shipments").to_frame())

flagged: 3330 of 166453 (2% by construction)


,flagged shipments
vendor,
V006,706
V002,410
V007,382
V001,241
V003,197


## Lane-vendor rate table (used by the app)

In [24]:
lv = lr[(lr["r2_freight"] > 0) & (lr["gta_invoice_value"] > 0) & (lr["lr_qty"] > 0)].copy()
lv["vehicle_type"] = lv["vehicle_type"].fillna("MISSING")
lane = lv.groupby(["route", "vehicle_type", "vendor_name"]).agg(lr_count=("delivery_id", "size"), freight=("r2_freight", "sum"),
        qty=("lr_qty", "sum"), margin=("margin", "sum"), gta=("gta_invoice_value", "sum")).reset_index()
lane = lane[lane.lr_count >= 10]
lane["rs_per_mt"] = lane["freight"] / lane["qty"]; lane["margin_pct"] = lane["margin"] / lane["gta"] * 100
lane[["route", "vehicle_type", "vendor_name", "lr_count", "rs_per_mt", "margin_pct"]].to_csv("../data/processed/lane_vendor_rates.csv", index=False)
print("lane_vendor_rates.csv rows:", len(lane))

lane_vendor_rates.csv rows: 3756


# Checkpoint summary

The cell below reads back the three metrics files that the report team will use.

In [25]:
import json
for t in ["V1", "V2", "V3"]:
    d = json.load(open(f"../reports/metrics/{t}.json", encoding="utf8"))
    print(f"\n=== {t}: {d['business_question']}  (winner: {d['winner']}, n_train={d['n_train']:,}, n_test={d['n_test']:,})")
    rows = {m["name"]: m["metrics"] for m in d["models"]}
    display(pd.DataFrame(rows).T)


=== V1: What should this truck movement cost, given only the lane and the load?  (winner: RandomForest, n_train=128,938, n_test=37,515)


,MAE,RMSE,MAPE,R2
Ridge,9825.00,21686.00,24.90,0.8051
RandomForest,2695.93,6691.31,11.29,0.9814
LightGBM,2730.40,6621.79,11.63,0.9818



=== V2: At booking, before the vendor is paid, which consignments will lose money?  (winner: LightGBM, n_train=219,109, n_test=65,312)


,PR_AUC,ROC_AUC,precision,recall,F1,threshold,val_precision_at_thr,val_recall_at_thr
Logistic,0.6617,0.8093,0.6264,0.5712,0.5975,0.7170,0.6,0.3535
RandomForest,0.6991,0.8632,0.6074,0.7989,0.6901,0.4904,0.6,0.7103
LightGBM,0.7429,0.8740,0.5967,0.8391,0.6975,0.5345,0.6,0.7407



=== V3: When planning, before loading, which lane and vehicle choices end up below 60% full?  (winner: LightGBM, n_train=124,252, n_test=37,186)


,PR_AUC,ROC_AUC,precision,recall,F1,threshold,val_precision_at_thr,val_recall_at_thr
Logistic,0.6616,0.9068,0.6558,0.5484,0.5973,0.8414,0.6001,0.5424
RandomForest,0.6899,0.9204,0.5417,0.6821,0.6038,0.4907,0.6001,0.6769
LightGBM,0.7516,0.9358,0.6554,0.6965,0.6753,0.6822,0.6000,0.7196
